# Notebook 14：时间参数化（Time Parameterization）

## 1. 本节在知识体系中的位置

```
NB15-16 运动规划 ──→ 路径 q(s) ──→ NB14 时间参数化 ──→ q(t) ──→ NB17-18 控制
                       (几何)          (加时间律)      (时变参考)
```

运动规划器输出一条几何路径 $\mathbf{q}(s)$。时间参数化回答："沿着这条路径，什么时刻该走到哪儿？"——即找到 $s(t)$ 使 $\mathbf{q}(t) = \mathbf{f}(s(t))$ 满足所有关节的速度/加速度/力矩约束，且**总时间最短或尽量短**。

## 2. 学习目标

- ⭐ 理解路径参数化 $s \in [0,1]$ 和时间放缩 $\mathbf{q}(t) = \mathbf{f}(s(t))$
- ⭐ 掌握速度/加速度的链式分解
- ⭐ 理解 TOPP 的核心思想：在 $(s, \dot{s})$ 相平面上寻找可行域
- 📖 前向积分 + 后向积分求时间最优轨迹
- 📚 TOPP-RA 和力矩约束

## 3. 路径参数化的数学 ⭐

### 3.1 链式分解

给定几何路径 $\mathbf{q} = \mathbf{f}(s)$（由规划器提供，NB15-16），速度/加速度为：
$$\dot{\mathbf{q}} = \mathbf{f}'(s) \dot{s}$$
$$\ddot{\mathbf{q}} = \mathbf{f}'(s) \ddot{s} + \mathbf{f}''(s) \dot{s}^2$$

将关节约束转嫁到 $s$ 和 $\dot{s}$ 上：
$$-\dot{\mathbf{q}}_{max} \leq \mathbf{f}'(s) \dot{s} \leq \dot{\mathbf{q}}_{max}$$
$$-\ddot{\mathbf{q}}_{max} \leq \mathbf{f}'(s) \ddot{s} + \mathbf{f}''(s) \dot{s}^2 \leq \ddot{\mathbf{q}}_{max}$$

对于每个 $s$ 和每个关节 $i$，速度约束给出 $\dot{s}$ 的上界：
$$\dot{s} \leq \frac{\dot{q}_{max,i}}{|f'_i(s)|} \quad \text{（仅当 } f'_i(s) \neq 0 \text{ 时）}$$

### 3.2 (s, ṡ) 相平面

将路径问题转化为二维 $(s, \dot{s})$ 空间中的问题：
- 横轴 $s \in [0,1]$：沿路径的进度
- 纵轴 $\dot{s}$：进度速率
- 可行域 = 满足所有关节速度/加速度约束的 $(s, \dot{s})$ 区域
- MVC（Maximum Velocity Curve）= 可行域的上边界

## 4. TOPP 算法核心思想

### 4.1 时间最优解的特征

时间最优轨迹一定贴着约束边界运动（bang-bang 原理）：
- 要么以最大速度运动（在 MVC 上）
- 要么以最大加速度加速、最大减速度减速

### 4.2 两遍积分

1. **前向积分**（从 $s=0$ 开始，$\dot{s}=0$）：用最大加速度加速，不超出 MVC
2. **后向积分**（从 $s=1$ 开始，$\dot{s}=0$）：用最大减速度减速，不超出 MVC
3. 两条曲线围成的下包络 = 时间最优 $\dot{s}(s)$ 曲线

## 5. Python 实现

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
%matplotlib inline
print("✅ 导入完成")

### 5.1 2R 臂路径的 MVC 计算

In [ ]:
# 定义一条从 q_start 到 q_goal 的直线路径（在关节空间）
q_start = np.array([0.0, 0.0])
q_goal = np.array([np.pi/2, np.pi/3])
n_s = 200
s_grid = np.linspace(0, 1, n_s)

# 路径 f(s) = (1-s)*q_start + s*q_goal (直线)
def path(s):
    return (1-s)*q_start + s*q_goal

def path_deriv(s):
    return q_goal - q_start  # f'(s) = constant for linear path

# 计算 MVC（速度约束）
q_dot_max = np.array([3.0, 4.0])  # 各关节最大速度
s_dot_max_speed = np.full(n_s, np.inf)

for i in range(n_s):
    fp = path_deriv(s_grid[i])
    for d in range(2):
        if abs(fp[d]) > 1e-10:
            s_dot_max_speed[i] = min(s_dot_max_speed[i], q_dot_max[d] / abs(fp[d]))

# 加速度约束简化处理
q_ddot_max = np.array([8.0, 10.0])

### 5.2 前向 + 后向积分

In [ ]:
# 简化版 TOPP：从前向后和从后向前扫描
s_dot_fwd = np.zeros(n_s)
s_dot_fwd[0] = 0.0

# 前向：用加速度上限加速
for i in range(1, n_s):
    ds = s_grid[i] - s_grid[i-1]
    # 最大允许加速度产生的速率
    s_ddot_max_val = np.inf
    fp = path_deriv(s_grid[i])
    for d in range(2):
        if abs(fp[d]) > 1e-10:
            # s̈ ≤ (q̈_max - f'' ṡ²)/f'（此处 f''=0 因为直线路径）
            s_ddot_max_val = min(s_ddot_max_val, q_ddot_max[d] / abs(fp[d]))
    # 欧拉积分
    s_dot_possible = s_dot_fwd[i-1] + s_ddot_max_val * ds / max(s_dot_fwd[i-1], 1e-3)
    s_dot_fwd[i] = min(s_dot_possible, s_dot_max_speed[i])

# 后向
s_dot_bwd = np.zeros(n_s)
s_dot_bwd[-1] = 0.0
for i in range(n_s-2, -1, -1):
    ds = s_grid[i+1] - s_grid[i]
    s_ddot_max_val = np.inf
    fp = path_deriv(s_grid[i])
    for d in range(2):
        if abs(fp[d]) > 1e-10:
            s_ddot_max_val = min(s_ddot_max_val, q_ddot_max[d] / abs(fp[d]))
    s_dot_possible = s_dot_bwd[i+1] + s_ddot_max_val * ds / max(s_dot_bwd[i+1], 1e-3)
    s_dot_bwd[i] = min(s_dot_possible, s_dot_max_speed[i])

# 时间最优 = min(前向, 后向, MVC)
s_dot_opt = np.minimum(np.minimum(s_dot_fwd, s_dot_bwd), s_dot_max_speed)

# 对比均匀时间参数化
s_dot_uniform = np.full(n_s, np.mean(s_dot_opt[s_dot_opt > 0]))  # 常数速率

# 计算总时间
T_opt = np.trapezoid(1.0 / (s_dot_opt + 1e-6), s_grid)
T_uniform = np.trapezoid(1.0 / (s_dot_uniform + 1e-6), s_grid)

### 5.3 相平面可视化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# (s, ṡ) 相平面
ax1.fill_between(s_grid, 0, s_dot_max_speed, color='green', alpha=0.1, label='Feasible Region')
ax1.plot(s_grid, s_dot_max_speed, 'k-', linewidth=2, label='MVC (Max Velocity Curve)')
ax1.plot(s_grid, s_dot_fwd, 'b--', linewidth=1.5, label='Forward Pass')
ax1.plot(s_grid, s_dot_bwd, 'r--', linewidth=1.5, label='Backward Pass')
ax1.plot(s_grid, s_dot_opt, 'purple', linewidth=2.5, label='Optimal ṡ(s)')
ax1.plot(s_grid, s_dot_uniform, 'orange', linewidth=1.5, alpha=0.7, label='Uniform ṡ')
ax1.set_xlabel('Path Progress s'); ax1.set_ylabel('ṡ')
ax1.set_title('(s, ṡ) Phase Plane — TOPP')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# 时间对比
# 模拟两种参数化的 q₁(t) 轨迹
t_opt_vals = np.zeros(n_s)
t_unif_vals = np.zeros(n_s)
t_opt_vals[0] = 0; t_unif_vals[0] = 0
for i in range(1, n_s):
    ds = s_grid[i] - s_grid[i-1]
    t_opt_vals[i] = t_opt_vals[i-1] + ds / max(s_dot_opt[i], 1e-6)
    t_unif_vals[i] = t_unif_vals[i-1] + ds / s_dot_uniform[i]

q_opt = np.array([path(s) for s in s_grid])
ax2.plot(t_opt_vals, q_opt[:, 0], 'b-', linewidth=2, label=f'q₁ (Optimal, T={T_opt:.3f}s)')
ax2.plot(t_unif_vals, q_opt[:, 0], 'orange', linewidth=2, label=f'q₁ (Uniform, T={T_uniform:.3f}s)')
ax2.set_xlabel('Time t (s)'); ax2.set_ylabel('q₁ (rad)')
ax2.set_title(f'Time Comparison: Optimal saves {(T_uniform-T_opt)/T_uniform*100:.0f}% time')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/14_topp_phase_plane.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. 练习题

### 概念题
1. 路径 vs 轨迹 vs 时间参数化：三者的关系是什么？
2. MVC 曲线由什么约束决定？

### 编程题
1. 修改上述代码，将直线路径替换为圆弧路径（带 $f''$ 非零项）。
2. 加入简化的关节力矩约束。

> 答案见 `solutions/14_solutions.ipynb`

## 7. 本节总结

| 概念 | 公式 | 含义 |
|------|------|------|
| 链式分解 | $\ddot{\mathbf{q}} = \mathbf{f}'\ddot{s} + \mathbf{f}''\dot{s}^2$ | 路径导数 + 速率导数 |
| MVC | $\dot{s}_{max}(s)$ 从速度/加速度约束计算 | 可行域上界 |
| 时间最优 | 贴着 MVC 边界运动 | bang-bang 原理 |
| 总时间 | $T = \int_0^1 \frac{ds}{\dot{s}(s)}$ | ṡ 越大越快 |